In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -U langchain langchain-community langchain-text-splitters langchain-huggingface sentence-transformers chromadb langchain-chroma pymupdf opentelemetry-api opentelemetry-sdk
!pip install langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 10.7 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.3
    Uninstalling click-8.3.3:
      Successfully uninstalled click-8.3.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
deepeval 4.2.0 requires click<8.4.0,>=8.0.0, but you have click 8.5.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.


In [3]:
import os
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


/tmp/ipykernel_33580/2993614614.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader


Używam 5 plików pdf wyciągniętych z książki US Army Survival Manual FM 21-76 dotyczących instrukcji przetrwania jako kontekstu dla modelu: Dangerous animals.pdf, Firecraft.pdf, Plants.pdf, Shelters.pdf, Water procurement.pdf

In [4]:
pdf_path = "/content/drive/MyDrive/PdfFiles"

loader = DirectoryLoader(
    pdf_path,
    glob = "*.pdf",
    loader_cls= PyMuPDFLoader
    )

pages = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)
docs = text_splitter.split_documents(pages)

Chunk_size zwiększony do 800, żeby model dostawał całe akapity

In [5]:
print("Liczba załadowanych stron:", len(pages))
print("Liczba utworzonych fragmentów:", len(docs))

Liczba załadowanych stron: 82
Liczba utworzonych fragmentów: 189


In [6]:
for idx, doc in enumerate(docs[30:33]):
  print(f"[{idx}].{doc}\n\n")

[0].page_content='• Cover the top of the bed frame with broad leaves or grass
to form a soft sleeping surface.
• Build a fire pad by laying clay, silt, or mud on one corner of
the swamp bed and allow it to dry.
5-38. Another shelter designed to get you above and out of the
water or wet ground uses the same rectangular configuration as
the swamp bed. You simply lay sticks and branches lengthwise on
the inside of the trees (or poles) until there is enough material to
raise the sleeping surface above the water level.' metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': '2026-08-31T12:51:19+00:00', 'source': '/content/drive/MyDrive/PdfFiles/Shelters.pdf', 'file_path': '/content/drive/MyDrive/PdfFiles/Shelters.pdf', 'total_pages': 21, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20260831125119+00'00'", 'page': 14}


[1].page_content='FM 3-05.70
5-16
 NATURAL SHELTERS
5-39. Do n

In [7]:
import spacy
import re

nlp = spacy.load("en_core_web_sm")

for chunk in docs:

  text = chunk.page_content
  text = text.replace('\n', ' ')
  text = re.sub(r'(\w+)-\s+(\w+)', r'\1\2', text)
  text = re.sub(r'\s+', ' ', text)

  chunk.page_content = text.strip()

doc = nlp(docs[20].page_content)

print([token.text for token in doc])

['and', 'tie', 'it', 'securely', 'to', 'the', 'tree', 'trunk', '.', 'Figure', '5', '-', '6', '.', 'No', '-', 'Pole', 'Parachute', 'Tepee', 'ONE', '-', 'MAN', 'SHELTER', '5', '-', '27', '.', 'A', 'one', '-', 'man', 'shelter', '(', 'Figure', '5', '-', '7', ',', 'page', '5', '-', '11', ')', 'you', 'can', 'easily', 'make', 'using', 'a', 'parachute', 'requires', 'a', 'tree', 'and', 'three', 'poles', '.', 'One', 'pole', 'should', 'be', 'about', '4.5', 'meters', '(', '15', 'feet', ')', 'long', 'and', 'the', 'other', 'two', 'about', '3', 'meters', '(', '10', 'feet', ')', 'long', '.']


In [8]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [9]:
from langchain_chroma import Chroma

# vectorstore = Chroma.from_documents(
#     documents=docs,
#     embedding=embeddings,
#     persist_directory="/content/drive/MyDrive/Vector_store"
#     )

vectorstore = Chroma(
    embedding_function=embeddings,
    persist_directory="/content/drive/MyDrive/Vector_store"
    )

In [10]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
    )

In [11]:
similar = retriever.invoke('How to lay the pyramid fire?')
for doc in similar:
  print(doc.page_content[:300] + "...\n---")

to provide a draft. PYRAMID 7-17. To lay the pyramid fire (Figure 7-5), place two small logs or branches parallel on the ground. Place a solid layer of small logs across the parallel logs. Add three or four more layers of logs, each layer smaller than and at a right angle to the layer below it. Make...
---
FM 3-05.70 7-7 LEAN-TO 7-15. To lay a lean-to fire (Figure 7-5), push a green stick into the ground at a 30-degree angle. Point the end of the stick in the direction of the wind. Place some tinder deep under this lean-to stick. Lean pieces of kindling against the lean-to stick. Light the tinder. As ...
---
HOW TO BUILD A FIRE 7-13. There are several methods for laying a fire and each one has advantages. The situation you are in will determine which of the following fires to use. TEPEE 7-14. To make a tepee fire (Figure 7-5, page 7-7), arrange the tinder and a few sticks of kindling in the shape of a t...
---


In [12]:
from google.colab import userdata

api_key = userdata.get('Gemini_API_Key')

os.environ["GOOGLE_API_KEY"] = api_key

#Narzędzia dla agenta

In [13]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests numpy pandas

In [14]:
from langchain_core.tools import tool
from datetime import datetime

@tool
def search_knowledge_base(query: str) -> str:
    """Searches the knowledge base for information from the loaded PDF documents.
    Use this tool when the user asks about the content of the documents.
    The query argument must contain only the main keywords. Do not pass full questions like: 'Is it good to do this?'
    Call this tool exactly once per question.
    """
    print("DEBUG: search_knowledge_base used")

    docs = retriever.invoke(query)

    if not docs:
        return "No matching information in the knowledge base."

    results = []
    for i, doc in enumerate(docs):
      source = doc.metadata.get('source', 'Unknown source')
      page = doc.metadata.get('page', 'Unknown page')

      chunk = (
          f"Content: {doc.page_content}\n"
          f"Metadata_Source: {source}\n"
          f"Metadata_Page: {page}\n"
          f"--------------------\n"
        )

      results.append(chunk)

    return "\n\n---\n\n".join(results) + "\n\n"


import pandas as pd
import openmeteo_requests
import requests_cache
import requests
from retry_requests import retry

def detect_current_location() -> dict:
    """Detect the user's current city, country and GPS coordinates based on their IP address
    when the user asks about their location or when this information is required to be passed
    as an argument into other functions.
    """
    print("DEBUG: detect_current_location used")

    url = "http://ip-api.com/json/"
    response = requests.get(url, timeout=5).json()

    if response.get("status") == "success":
        return {
              "city": response.get("city"),
              "country": response.get("country"),
              "latitude": response.get("lat"),
              "longitude": response.get("lon")
            }
    else:
        return {"error": "Could not determine location from current IP."}


@tool
def get_weather_forecast(location: str = None, unit = "celsius"):
    """Get the weather forecast for a given location using the Open-Meteo API.
    If the user asks about weather without naming a specific location, leave the 'location' argument as None.
    Do not guess or assume the current location.
    """
    print("DEBUG: get_weather_forecast used")

    #Większość kodu wzięta z oficjalnej dokumentacji Open-Meteo:

    # Setup the Open-Meteo API client with cache and retry on error
    cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
    retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
    openmeteo = openmeteo_requests.Client(session = retry_session)

    if not location:
      current_data = detect_current_location()
      latitude = current_data["latitude"]
      longitude = current_data["longitude"]
    else:
      url = f"https://geocoding-api.open-meteo.com/v1/search?name={location}&count=1&language=en&format=json"
      response_geo = requests.get(url).json()

      if not response_geo.get("results"):
        return f"Location was not found: {location}"

      latitude = response_geo["results"][0]["latitude"]
      longitude = response_geo["results"][0]["longitude"]

    # The order of variables in hourly or daily is important to assign them correctly below
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
    "latitude": latitude,
    "longitude": longitude,
    "daily": ["weather_code", "temperature_2m_max", "temperature_2m_min", "precipitation_probability_max", "precipitation_hours"],
    "timezone": "auto",
    }
    responses = openmeteo.weather_api(url, params = params)

    # Process location.
    response = responses[0]
    coordinates = f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E"
    elevation = f"Elevation: {response.Elevation()} m asl"
    timezone_difference = f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s"

    # Process daily data. The order of variables needs to be the same as requested.
    daily = response.Daily()
    daily_weather_code = daily.Variables(0).ValuesAsNumpy()
    daily_temperature_2m_max = daily.Variables(1).ValuesAsNumpy()
    daily_temperature_2m_min = daily.Variables(2).ValuesAsNumpy()
    daily_precipitation_probability_max = daily.Variables(3).ValuesAsNumpy()
    daily_precipitation_hours = daily.Variables(4).ValuesAsNumpy()

    daily_data = {
      "date": pd.date_range(
        start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
        end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
        freq = pd.Timedelta(seconds = daily.Interval()),
        inclusive = "left"
      )
    }

    daily_data["weather_code"] = daily_weather_code
    daily_data["temperature_2m_max"] = daily_temperature_2m_max
    daily_data["temperature_2m_min"] = daily_temperature_2m_min
    daily_data["precipitation_probability_max"] = daily_precipitation_probability_max
    daily_data["precipitation_hours"] = daily_precipitation_hours

    weather_codes = {
    0: "Clear sky",
    1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
    45: "Fog", 48: "Depositing rime fog",
    51: "Drizzle: Light", 53: "Drizzle: Moderate", 55: "Drizzle: Dense intensity",
    56: "Freezing Drizzle: Light", 57: "Freezing Drizzle: Dense intensity",
    61: "Rain: Slight", 63: "Rain: Moderate", 65: "Rain: Heavy intensity",
    66: "Freezing Rain: Light", 67: "Freezing Rain: Heavy intensity",
    71: "Snow fall: Slight", 73: "Snow fall: Moderate", 75: "Snow fall: Heavy intensity",
    77: "Snow grains",
    80: "Rain showers: Slight", 81: "Rain showers: Moderate", 82: "Rain showers: Violent",
    85: "Snow showers: Slight", 86: "Snow showers: Heavy",
    95: "Thunderstorm: Slight or moderate",
    96: "Thunderstorm with slight hail", 99: "Thunderstorm with heavy hail"
    }

    forecast = {
        'coordinates': coordinates,
        'elevation': elevation,
        'timezone_difference': timezone_difference,
        'date': [day.strftime("%Y-%m-%d") for day in daily_data['date']],
        'weather': [weather_codes.get(code) for code in daily_data["weather_code"]],
        'temperature_2m_max': daily_data["temperature_2m_max"].tolist(),
        'temperature_2m_min': daily_data["temperature_2m_min"].tolist(),
        'precipitation_probability_max': daily_data["precipitation_probability_max"].tolist(),
        'precipitation_hours': daily_data["precipitation_hours"].tolist()
    }

    return forecast

In [15]:
tools = [detect_current_location, get_weather_forecast, search_knowledge_base]

#Tworzenie agenta

In [16]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig

In [17]:
agent = create_agent(
    model= "google_genai:gemini-3.1-flash-lite",
    tools=tools,
    system_prompt="""You are an expert AI assistant specializing in survival.
    Rules:
    1. Answer the user's question using only the provided base context. Do not use pre-trained knowledge to guess answers.

    2. Every time you don't have information about the question that the user asked you, say: There is no information in the knowledge base.
    Do not make up answers. However, you are allowed to draw direct logical conclusions from the facts (e.g. if the animal is poisonous it is not safe).

    3. Before finalizing your answer, check the conversational history.

    If the user uses pronouns (he, she, it, they, him, her, etc.),
    replace them with the specific nouns they refer to from the past messages.
    Input: I moved into a new house with my family. It is large and cozy. They really loved our new house. We are happy to be here.
    Output: I moved into a new house with my family. The house is large and cozy. The family really loved our new house. My family and I are happy to be here.

    If the current question uses plural words (they, both, them, etc.), you must group the main topics from all previous messages and combine them together into your search query.
    Do not look only at the last message.

    4. Be concise. Focus on the direct answer to the user's question and ignore any extra background facts.

    5. Never include full folder paths, absolute paths or drive paths when reffering to sources, you must only provide the exact filename.""",

    checkpointer = InMemorySaver()
    )

In [18]:
config: RunnableConfig = {
    "configurable": {"thread_id":"10000"}
}

In [19]:
from langchain_core.messages import ToolMessage

In [20]:
def parse_result(res):
  messages = res['messages']
  answer = res['messages'][-1].content[0]['text']

  context = []
  retrieved_docs = []
  for message in reversed(messages):
    if isinstance(message, ToolMessage):
      if message.name == "search_knowledge_base":

        content = message.content
        context.append(str(content))

        sources = re.findall(r"Source:\s*(?:.*/)?([^,\n]+\.pdf)", content)
        retrieved_docs.extend(sources)

        break

  return {"answer" : answer,
          "context": context,
          "retrieved_docs": retrieved_docs}

In [21]:
def run_agent(query: str, config: RunnableConfig):
  config["recursion_limit"] = 10
  result = agent.invoke({'messages':[{'role':'user', 'content': query}]}, config)
  return parse_result(result)

In [22]:
result_list = list()

result_list.append(run_agent("What should I do when I encounter a snake?", config))
result_list.append(run_agent("Is chinaberry save to eat?", config))
result_list.append(run_agent("Are they both poisonous?", config))

for res in result_list:
  print(res["answer"])

DEBUG: search_knowledge_base used
DEBUG: search_knowledge_base used
DEBUG: search_knowledge_base used
When you encounter a snake, you should remain calm. Remember that snakes cannot hear, so they may be surprised if you encounter them while they are sleeping or sunning. Normally, snakes will flee if given the opportunity. Do not tease, molest, or harass them, as some species will attack aggressively when cornered or guarding a nest. If you must kill a snake for food or safety, use extreme care.
Chinaberry is not safe to eat; it is listed as a plant that can cause ingestion poisoning if eaten.
Chinaberry is poisonous and can cause ingestion poisoning if eaten. Regarding snakes, while many are not venomous, the knowledge base identifies several as venomous (such as the American Copperhead, Bushmaster, Coral snake, Cottonmouth, Fer-de-lance, Rattlesnake, Common adder, Pallas’ viper, Boomslang, and Cobra). Therefore, it is not accurate to classify all snakes as poisonous, but many species 

Model pamięta poprzednią część konwersacji oraz rozumie zaimki (rozumie, że both to są roślina i wąż), a słowo poisonous go nie zmyliło. Model zrozumiał, że chodzi ogólnie o toksyny.

In [23]:
result_list = list()

result_list.append(run_agent("What weather is in California?", config))
result_list.append(run_agent("What will it be tommorow?", config))
result_list.append(run_agent("And the next day after tommorow?", config))

for res in result_list:
  print(res["answer"])

DEBUG: get_weather_forecast used
The weather in California over the next week will be as follows:

*   **September 2nd:** Overcast, with a high of 37.3°C and a low of 24.5°C.
*   **September 3rd:** Overcast, with a high of 37.3°C and a low of 25.4°C.
*   **September 4th:** Overcast, with a high of 39.6°C and a low of 24.6°C.
*   **September 5th:** Overcast, with a high of 39.4°C and a low of 27.0°C.
*   **September 6th:** Moderate drizzle, with a high of 40.5°C and a low of 22.3°C.
*   **September 7th:** Overcast, with a high of 40.4°C and a low of 21.3°C.
*   **September 8th:** Partly cloudy, with a high of 42.5°C and a low of 24.6°C.
The weather tomorrow, September 3rd, will be overcast with a high of 37.3°C and a low of 25.4°C.
The weather the day after tomorrow, September 4th, will be overcast with a high of 39.6°C and a low of 24.6°C.


Pamięta, że chodzi o Californię i podaje prognozę pogody dla różnych dni

In [24]:
result_list = list()

result_list.append(run_agent("What is the weather now in my city?", config))
result_list.append(run_agent("For which location is this forecast?", config))

for res in result_list:
  print(res["answer"])

DEBUG: detect_current_location used
DEBUG: get_weather_forecast used
The weather in your city, Singapore, today, September 1st, is light drizzle with a high of 33.4°C and a low of 27.5°C.
This forecast is for Singapore, which was detected as your current city.


Potrafi wykryć lokalizację użytkownika i podać prognozę pogody dla danej miejscowości. Poniżej jest podane IP tego collaba:

In [25]:
!curl ifconfig.me

34.21.168.162

In [26]:
url = f"http://ip-api.com/json/34.21.168.162"
response = requests.get(url).json()
response


{'status': 'success',
 'country': 'Singapore',
 'countryCode': 'SG',
 'region': '01',
 'regionName': 'Central Singapore',
 'city': 'Singapore',
 'zip': '190464',
 'lat': 1.3048,
 'lon': 103.8622,
 'timezone': 'Asia/Singapore',
 'isp': 'Google LLC',
 'org': 'Google Cloud (asia-southeast1)',
 'as': 'AS396982 Google LLC',
 'query': '34.21.168.162'}

In [27]:
result_list = list()

result_list.append(run_agent("What animals are venomous?", config))
result_list.append(run_agent("What is the source?", config))
result_list.append(run_agent("how to navigate in the woods without a compass? Give a source", config)) #tej informacji nie ma w bazie danych


for res in result_list:
  print(res["answer"])

DEBUG: search_knowledge_base used
DEBUG: search_knowledge_base used
The venomous animals identified in the knowledge base include:

*   **Snakes:** American Copperhead, Bushmaster, Coral snake, Cottonmouth, Fer-de-lance, Rattlesnake, Common adder, Pallas’ viper, Boomslang, and Cobra.
*   **Other:** Bees (mentioned for their stings).

The knowledge base also notes that nature has given many small animals weapons such as fangs and stingers to defend themselves.
The information provided is sourced from the document: **Dangerous animals.pdf**.
There is no information in the knowledge base regarding how to navigate in the woods without a compass.


Podaje źródło i jeśli nie znajduje informacji w bazie danych, informuje o tym, a nie zmyśla

# Metryki

In [28]:
def calculate_hit_rate_at_k(relevant_docs, retrieved_docs, k):
  """Calculate Hit Rate@k"""
  top_k_retrieved = set(retrieved_docs[:k])
  return 1.0 if any(doc in top_k_retrieved for doc in relevant_docs) else 0.0

In [29]:
def calculate_mrr(relevant_docs, retrieved_docs):
  """Calculate Mean Reciprocal Rank"""
  for position, doc_id in enumerate(retrieved_docs, 1):
    if doc_id in relevant_docs:
      return 1.0 / position

  return 0.0 #No relevant document found

In [30]:
import time

Pytania dla ewaluacji modelu zostały wygenerowane przez AI i następnie zmodyfikowane:

In [31]:
config_eval: RunnableConfig = {
    "configurable": {"thread_id": "eval"}
}

In [32]:
eval_questions = [

     # Dangerous Animals
     {"question":"Which smaller animals cause more annual deaths than large predators?", "relevant_doc": "Dangerous animals.pdf"},
     {"question":"How can you recognize a brown recluse spider, and what can its bite cause?", "relevant_doc": "Dangerous animals.pdf"},
     {"question":"How long must a tick remain attached to a host before it transmits disease?", "relevant_doc": "Dangerous animals.pdf"},
     {"question":"What is the danger of taking shelter in a cave occupied by bats?", "relevant_doc": "Dangerous animals.pdf"},

     # Firecraft
     {"question":"What are the components of the fire triangle?", "relevant_doc": "Firecraft.pdf"},
     {"question":"What type of rocks must you avoid using when constructing a fire wall and why?","relevant_doc": "Firecraft.pdf"},
     {"question":"What is charred cloth, and how is it prepared for a survival kit?","relevant_doc": "Firecraft.pdf"},
     {"question":"What are the items needed to start a fire using the primitive method?","relevant_doc": "Firecraft.pdf"},

     # Plants
     {"question":"What type of plant should you never harvest when looking for food?","relevant_doc": "Plants.pdf"},
     {"question":"Which plant provides a raw component for pain relief?","relevant_doc": "Plants.pdf"},
     {"question":"How plants can poison a human body?","relevant_doc": "Plants.pdf"},
     {"question":"What should you do if you experience the symptoms of contact dermatitis?", "relevant_doc": "Plants.pdf"},

     # Shelters
     {"question":"What does the BLISS stand for?","relevant_doc": "Shelters.pdf"},
     {"question":"What is a drip stick, and how does it function on a shelter?","relevant_doc": "Shelters.pdf"},
     {"question":"How much of your body heat can you lose to the ground when at rest?","relevant_doc": "Shelters.pdf"},
     {"question":"What type of shelter is the best to keep you dry when camping in standing water environment?","relevant_doc": "Shelters.pdf"},

     # Water Procurement
     {"question":"Why is it dangerous to eat unmelted snow or ice?","relevant_doc": "Water procurement.pdf"},
     {"question":"How can you harvest palatable water from a banana and for how long will it supply water?","relevant_doc": "Water procurement.pdf"},
     {"question":"What is the most effective method of destroying all waterborne pathogens?","relevant_doc": "Water procurement.pdf"},
     {"question":"Why are some common chemical disinfectants are not considered effective?","relevant_doc": "Water procurement.pdf"},
 ]

eval_results = []
for item in eval_questions:
    q = item['question']
    relevant_doc = item["relevant_doc"]
    res = run_agent(q, config_eval)
    res["question"] = q
    res["relevant_docs"] = [relevant_doc]
    eval_results.append(res)
    print(f"Q: {q}")
    print(f"A: {res['answer']}")
    print()

    time.sleep(5)



DEBUG: search_knowledge_base used
Q: Which smaller animals cause more annual deaths than large predators?
A: According to the document *Dangerous animals.pdf*, smaller animals that cause more annual deaths than large dangerous animals are relatively small venomous snakes and bees, which cause deaths due to allergic reactions to bee stings.

DEBUG: search_knowledge_base used
Q: How can you recognize a brown recluse spider, and what can its bite cause?
A: The brown recluse (or fiddleback spider) is recognized by a prominent, violin-shaped light spot on its back. If it bites, it can cause excessive tissue degeneration around the wound, which may lead to the amputation of digits if left untreated.

DEBUG: search_knowledge_base used
Q: How long must a tick remain attached to a host before it transmits disease?
A: According to *Dangerous animals.pdf*, a tick must be attached to a host for at least 6 hours to transmit disease organisms.

DEBUG: search_knowledge_base used
Q: What is the danger

Jedyne problemy: czasami model podaje więcej informacji niż go pytano (kiedy w system prompcie ignorowanie faktów niezwiązanych z pytaniem użytkownika znajdowało się wyżej, było lepiej). Również, model czasami podaje źródło, a czasami nie (gdy dodawałam system prompt o tym, żeby podawał źródło tylko wtedy, gdy użytkownik o niego pyta, dodawał źródło pod każdą odpowiedzią). No i także, w pytaniu o drip stick powyżej nie podał odpowiedzi, chociaż ta informacja znajduje się w jego bazie danych i czasami udawało mu się na to pytanie odpowiedzieć.

In [33]:
mrr = []
hit_rate_3 = []
hit_rate_4 = []
hit_rate_5 = []

for item in eval_results:

     retrieved_docs = item["retrieved_docs"]
     relevant_docs = item["relevant_docs"]

     mrr_val = calculate_mrr(relevant_docs,retrieved_docs)
     hit_rate_val_3 = calculate_hit_rate_at_k(relevant_docs,retrieved_docs,k=3)

     mrr.append(mrr_val)
     hit_rate_3.append(hit_rate_val_3)


print(f"Mean MRR: {sum(mrr) / len(mrr):.4f}")
print(f"Mean Hit Rate@3: {sum(hit_rate_3) / len(hit_rate_3):.4f}")

Mean MRR: 0.9417
Mean Hit Rate@3: 1.0000


Agent znajduje właściwy plik w pierwszej trójce wyników. Wynik MRR wskazuje, że w większości przypadków poprawny dokument znajduje się na pierwszej pozycji.